In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
print("start")


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
Visible GPUs: 2
GPU name: Tesla T4


In [3]:
!pip uninstall -y protobuf
!pip install protobuf==3.20.3

Found existing installation: protobuf 5.29.5
Uninstalling protobuf-5.29.5:
  Successfully uninstalled protobuf-5.29.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.2 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
onnx 1.20.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
ray 2.52.1 requires click!=8.3.*,>=7.0, but you have click 8.3.1 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
ydf 0.13.0 

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import transformers

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    TrainingArguments,
    Trainer,
    logging
)

path = "/kaggle/working/state.db"
if os.path.exists(path):
    os.remove(path)
    print("state.db deleted")
else:
    print("state.db not found")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    macro_f1 = f1_score(labels, predictions, average="binary")
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average="binary")
    recall = recall_score(labels, predictions, average="binary")

    return {
        "macro_f1": macro_f1,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall
    }

base_path = "/kaggle/input/sem-eval-2026-task-13-subtask-a/Task_A"

training_path = base_path + "/train.parquet"
validation_path = base_path + "/validation.parquet"
test_sample_path = base_path + "/test_sample.parquet"
test_path = base_path + "/test.parquet"
sample_sub_path = base_path + "/sample_submission.csv"

training_df = pd.read_parquet(training_path)
validation_df = pd.read_parquet(validation_path)
test_sample_df = pd.read_parquet(test_sample_path)
test_df = pd.read_parquet(test_path)
sample_sub_df = pd.read_csv(sample_sub_path)

for df in [training_df, validation_df, test_sample_df, test_df]:
    if "__index_level_0__" in df.columns:
        df.drop(columns=["__index_level_0__"], inplace=True)

print("Loaded shapes:")
print("train:", training_df.shape)
print("val:", validation_df.shape)
print("test:", test_df.shape)
print("test_sample:", test_sample_df.shape)
print("sample_submission:", sample_sub_df.shape)

test_sample_merged = pd.merge(test_df, test_sample_df, on="code", how="inner")
print("test_sample_merged:", test_sample_merged.shape)

pretrained_model = "microsoft/unixcoder-base"
tokenizer = AutoTokenizer.from_pretrained(pretrained_model)

def preprocess_function(examples):
    return tokenizer(examples["code"], truncation=True, max_length=256)

training_dataset = Dataset.from_pandas(training_df)
validation_dataset = Dataset.from_pandas(validation_df)
test_dataset = Dataset.from_pandas(test_df)
test_sample_merged_dataset = Dataset.from_pandas(test_sample_merged)

training_tokenized_set = training_dataset.map(preprocess_function, batched=True)
validation_tokenized_set = validation_dataset.map(preprocess_function, batched=True)
test_tokenized_set = test_dataset.map(preprocess_function, batched=True)
test_sample_merged_tokenized_set = test_sample_merged_dataset.map(preprocess_function, batched=True)

training_tokenized_set.set_format("torch")
validation_tokenized_set.set_format("torch")
test_tokenized_set.set_format("torch")
test_sample_merged_tokenized_set.set_format("torch")

model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model,
    num_labels=2
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("\nRunning on device:", device)

logging.set_verbosity_info()

class TrainingProgressCallback(transformers.TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.is_local_process_zero:
            print(f"Step {state.global_step}/{state.max_steps}")

OUTPUT_DIR = "/kaggle/working/checkpoints_task_a"
LOG_DIR = "/kaggle/working/logs_task_a"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    seed=42,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    metric_for_best_model="macro_f1",
    load_best_model_at_end=True,
    logging_dir=LOG_DIR,
    logging_steps=3000,
    fp16=True,
    disable_tqdm=False,
    ddp_find_unused_parameters=False,
    dataloader_num_workers=0,
    no_cuda=False,
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_tokenized_set,
    eval_dataset=validation_tokenized_set,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2),
               TrainingProgressCallback()]
)

resume = False
if os.path.isdir(OUTPUT_DIR):
    from transformers.trainer_utils import get_last_checkpoint
    last_ckpt = get_last_checkpoint(OUTPUT_DIR)
    if last_ckpt is not None:
        resume = True
        print("Found checkpoint:", last_ckpt)

if resume:
    trainer.train(resume_from_checkpoint=last_ckpt)
else:
    trainer.train()

print("Eval on validation:", trainer.evaluate(validation_tokenized_set))
print("Eval on test_sample_merged (labeled subset):", trainer.evaluate(test_sample_merged_tokenized_set))

predictions = trainer.predict(test_tokenized_set)
logits = predictions.predictions
predicted_labels = np.argmax(logits, axis=1).astype(int)

pred_df = pd.DataFrame({
    "ID": test_df["ID"].values,
    "label": predicted_labels
})

submission_df = sample_sub_df[["ID"]].merge(pred_df, on="ID", how="left")
missing = submission_df["label"].isna().sum()
print("Missing labels after merge:", missing)
submission_df["label"] = submission_df["label"].fillna(0).astype(int)

out_path = "/kaggle/working/submission.csv"
submission_df.to_csv(out_path, index=False)
print("Saved:", out_path)
submission_df.head()


2026-01-04 00:35:22.978910: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767486923.452550      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767486923.592038      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767486924.835958      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767486924.836006      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767486924.836009      55 computation_placer.cc:177] computation placer alr

state.db not found
Loaded shapes:
train: (500000, 4)
val: (100000, 4)
test: (1000, 2)
test_sample: (1000, 4)
sample_submission: (1000, 2)
test_sample_merged: (1000, 5)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

The following columns in the Evaluation set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: language, generator, ID, code. If language, generator, ID, code are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 16


Eval on validation: {'eval_loss': 0.024901509284973145, 'eval_macro_f1': 0.9939207402167887, 'eval_accuracy': 0.99364, 'eval_precision': 0.993844742224686, 'eval_recall': 0.993996749832712, 'eval_runtime': 1022.9962, 'eval_samples_per_second': 97.752, 'eval_steps_per_second': 6.11, 'epoch': 3.0}


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
The following columns in the test set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: __index_level_0__, ID, code. If __index_level_0__, ID, code are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 1000
  Batch size = 16


Eval on test_sample_merged (labeled subset): {'eval_loss': 5.335013389587402, 'eval_macro_f1': 0.3435251798561151, 'eval_accuracy': 0.27, 'eval_precision': 0.21484814398200225, 'eval_recall': 0.8565022421524664, 'eval_runtime': 10.2865, 'eval_samples_per_second': 97.215, 'eval_steps_per_second': 6.125, 'epoch': 3.0}
Missing labels after merge: 0
Saved: /kaggle/working/submission.csv


,ID,label
0,2005,1
1,2384,0
2,3526,0
3,3926,0
4,7222,1


In [1]:
print("test_sample_df:", test_sample_df.shape)
print("test_sample_merged:", test_sample_merged.shape)

print("Unique IDs in test_sample_df:", test_sample_df["ID"].nunique())
print("Unique IDs in test_sample_merged:", test_sample_merged["ID"].nunique())


test_sample_df: (1000, 4)
test_sample_merged: (1000, 5)


KeyError: 'ID'

In [2]:
print("train cols:", training_df.columns.tolist())
print("val cols:", validation_df.columns.tolist())
print("test cols:", test_df.columns.tolist())
print("test_sample cols:", test_sample_df.columns.tolist())


train cols: ['code', 'generator', 'label', 'language']
val cols: ['code', 'generator', 'label', 'language']
test cols: ['ID', 'code']
test_sample cols: ['code', 'generator', 'label', 'language']


In [3]:
train_g = set(training_df["generator"].unique())
val_g = set(validation_df["generator"].unique())
ts_g = set(test_sample_df["generator"].unique())

print("train generators:", len(train_g))
print("val generators:", len(val_g))
print("test_sample generators:", len(ts_g))

print("generators in test_sample but not in train:", sorted(ts_g - train_g)[:50])
print("generators in val but not in train:", sorted(val_g - train_g)[:50])


train generators: 35
val generators: 35
test_sample generators: 63
generators in test_sample but not in train: ['GPT-4o', 'Human', 'Qwen/Qwen2.5-72B-Instruct', 'Qwen/Qwen2.5-Codder-14B-Instruct', 'deepseek-ai/DeepSeek-V3-0324', 'gemini-1.5-flash', 'gemini-1.5-flash-8b', 'gemini-2.0-flash', 'gemini-2.0-flash-lite', 'gemini-2.5-flash-preview-05-20', 'gemma-3-27b-it', 'google/codegemma-7b-it', 'google/gemma-3-12b-it', 'google/gemma-3-27b-it', 'google/gemma-3-4b-it', 'meta-llama/Llama-3.2-11B-Vision-Instruct', 'meta-llama/Llama-3.2-90B-Vision-Instruct', 'meta-llama/Llama-3.3-70B-Instruct-Turbo', 'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8', 'meta-llama/Llama-4-Maverick-17B-128E-Instruct-Turbo', 'meta-llama/Llama-4-Scout-17B-16E-Instruct', 'meta-llama/Meta-Llama-3.1-405B-Instruct', 'meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo', 'microsoft/Phi-4-multimodal-instruct', 'microsoft/phi-4', 'mistralai/Devstral-Small-2505', 'mistralai/Mistral-7B-Instruct-v0.3', 'mistralai/Mistral-Nemo-Inst

In [4]:
# ============================================================
# Re-evaluate a saved UniXcoder checkpoint + generate submission
# (NO retraining)
# - Loads best/last checkpoint from OUTPUT_DIR
# - Evaluates on: validation.parquet and test_sample.parquet (labeled)
# - Optionally tunes a probability threshold on test_sample
# - Generates submission on test.parquet
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from transformers.trainer_utils import get_last_checkpoint

# ----------------------------
# Paths (edit if needed)
# ----------------------------
BASE_PATH = "/kaggle/input/sem-eval-2026-task-13-subtask-a/Task_A"
TRAIN_PATH = os.path.join(BASE_PATH, "train.parquet")
VAL_PATH = os.path.join(BASE_PATH, "validation.parquet")
TEST_SAMPLE_PATH = os.path.join(BASE_PATH, "test_sample.parquet")
TEST_PATH = os.path.join(BASE_PATH, "test.parquet")

# Where your checkpoints were saved during training
OUTPUT_DIR = "/kaggle/working/checkpoints_task_a"

# Output submission paths
SUBMISSION_ARGMAX_PATH = "/kaggle/working/submission_argmax.csv"
SUBMISSION_THRESH_PATH = "/kaggle/working/submission_thresholded.csv"

# Model + tokenization settings (must match training)
PRETRAINED_MODEL = "microsoft/unixcoder-base"
MAX_LENGTH = 256
BATCH_SIZE = 8

# ----------------------------
# Load dataframes
# ----------------------------
training_df = pd.read_parquet(TRAIN_PATH)
validation_df = pd.read_parquet(VAL_PATH)
test_sample_df = pd.read_parquet(TEST_SAMPLE_PATH)
test_df = pd.read_parquet(TEST_PATH)

for df in [training_df, validation_df, test_sample_df, test_df]:
    if "__index_level_0__" in df.columns:
        df.drop(columns=["__index_level_0__"], inplace=True)

print("Loaded shapes:")
print("train:", training_df.shape)
print("val:", validation_df.shape)
print("test_sample:", test_sample_df.shape)
print("test:", test_df.shape)
print("test columns:", test_df.columns.tolist())

# ----------------------------
# Quick generator shift summary (optional but useful)
# ----------------------------
train_g = set(training_df["generator"].unique()) if "generator" in training_df.columns else set()
val_g = set(validation_df["generator"].unique()) if "generator" in validation_df.columns else set()
ts_g = set(test_sample_df["generator"].unique()) if "generator" in test_sample_df.columns else set()

if train_g and ts_g:
    unseen = sorted(ts_g - train_g)
    print(f"\nGenerators: train={len(train_g)}, val={len(val_g)}, test_sample={len(ts_g)}")
    print(f"Unseen generators in test_sample (not in train): {len(unseen)}")
    print("First 25 unseen:", unseen[:25])

# ----------------------------
# Load checkpoint (NO training)
# ----------------------------
ckpt = get_last_checkpoint(OUTPUT_DIR)
if ckpt is None:
    raise FileNotFoundError(
        f"No checkpoint found under {OUTPUT_DIR}. "
        "Make sure training finished and saved checkpoints."
    )

print("\nLoading checkpoint:", ckpt)

tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=2)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Device:", device)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
    return tokenizer(examples["code"], truncation=True, max_length=MAX_LENGTH)

def df_to_tokenized_dataset(df: pd.DataFrame, has_labels: bool):
    ds = Dataset.from_pandas(df)
    if has_labels:
        # Make label column explicit for Trainer/prediction code below
        if "label" in ds.column_names and "labels" not in ds.column_names:
            ds = ds.rename_column("label", "labels")
    ds = ds.map(preprocess_function, batched=True)
    ds.set_format("torch")
    return ds

# Tokenized datasets
val_tok = df_to_tokenized_dataset(validation_df, has_labels=True)
ts_tok = df_to_tokenized_dataset(test_sample_df, has_labels=True)
test_tok = df_to_tokenized_dataset(test_df, has_labels=False)

# Minimal eval args
args = TrainingArguments(
    output_dir="/kaggle/working/tmp_eval",
    per_device_eval_batch_size=BATCH_SIZE,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# ----------------------------
# Evaluation helper
# ----------------------------
def eval_and_print(name: str, tokenized_ds):
    pred = trainer.predict(tokenized_ds)
    logits = pred.predictions
    y_true = np.array(tokenized_ds["labels"])

    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    y_hat = np.argmax(probs, axis=1)

    out = {
        "accuracy": accuracy_score(y_true, y_hat),
        "f1_binary": f1_score(y_true, y_hat, average="binary"),
        "f1_macro": f1_score(y_true, y_hat, average="macro"),
        "precision": precision_score(y_true, y_hat, average="binary", zero_division=0),
        "recall": recall_score(y_true, y_hat, average="binary", zero_division=0),
    }
    print(f"\n{name} metrics:")
    for k, v in out.items():
        print(f"  {k}: {v:.6f}")
    return out, probs, y_true

# Evaluate on validation and labeled test_sample
val_metrics, _, _ = eval_and_print("VALIDATION", val_tok)
ts_metrics, ts_probs, ts_y = eval_and_print("TEST_SAMPLE (labeled subset)", ts_tok)

# ----------------------------
# Threshold tuning on test_sample (optional, but often helps OOD)
# ----------------------------
def find_best_threshold(y_true, p_pos, metric="f1_binary"):
    best_t, best_score = 0.5, -1.0
    # coarse grid; increase resolution if you want
    for t in np.linspace(0.05, 0.95, 19):
        y_pred = (p_pos >= t).astype(int)
        if metric == "f1_macro":
            score = f1_score(y_true, y_pred, average="macro")
        else:
            score = f1_score(y_true, y_pred, average="binary")
        if score > best_score:
            best_score, best_t = score, float(t)
    return best_t, best_score

p_ai_ts = ts_probs[:, 1]
best_t, best_f1 = find_best_threshold(ts_y, p_ai_ts, metric="f1_binary")
print(f"\nBest threshold on TEST_SAMPLE for F1(binary): t={best_t:.2f}, F1={best_f1:.6f}")

# ----------------------------
# Predict on real TEST and save submissions
# ----------------------------
test_pred = trainer.predict(test_tok)
test_logits = test_pred.predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=1).numpy()

# 1) Argmax submission (what you already did)
test_labels_argmax = np.argmax(test_probs, axis=1).astype(int)
sub_argmax = pd.DataFrame({"ID": test_df["ID"].values, "label": test_labels_argmax})
sub_argmax.to_csv(SUBMISSION_ARGMAX_PATH, index=False)
print("\nSaved:", SUBMISSION_ARGMAX_PATH)
print(sub_argmax.head())

# 2) Thresholded submission (uses tuned threshold from test_sample)
test_labels_thresh = (test_probs[:, 1] >= best_t).astype(int)
sub_thresh = pd.DataFrame({"ID": test_df["ID"].values, "label": test_labels_thresh})
sub_thresh.to_csv(SUBMISSION_THRESH_PATH, index=False)
print("\nSaved:", SUBMISSION_THRESH_PATH)
print(sub_thresh.head())

print("\nDone. Submit one (or both) of these:")
print(" -", SUBMISSION_ARGMAX_PATH)
print(" -", SUBMISSION_THRESH_PATH)


Loaded shapes:
train: (500000, 4)
val: (100000, 4)
test_sample: (1000, 4)
test: (500000, 2)
test columns: ['ID', 'code']

Generators: train=35, val=35, test_sample=63
Unseen generators in test_sample (not in train): 30
First 25 unseen: ['GPT-4o', 'Human', 'Qwen/Qwen2.5-72B-Instruct', 'Qwen/Qwen2.5-Codder-14B-Instruct', 'deepseek-ai/DeepSeek-V3-0324', 'gemini-1.5-flash', 'gemini-1.5-flash-8b', 'gemini-2.0-flash', 'gemini-2.0-flash-lite', 'gemini-2.5-flash-preview-05-20', 'gemma-3-27b-it', 'google/codegemma-7b-it', 'google/gemma-3-12b-it', 'google/gemma-3-27b-it', 'google/gemma-3-4b-it', 'meta-llama/Llama-3.2-11B-Vision-Instruct', 'meta-llama/Llama-3.2-90B-Vision-Instruct', 'meta-llama/Llama-3.3-70B-Instruct-Turbo', 'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8', 'meta-llama/Llama-4-Maverick-17B-128E-Instruct-Turbo', 'meta-llama/Llama-4-Scout-17B-16E-Instruct', 'meta-llama/Meta-Llama-3.1-405B-Instruct', 'meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo', 'microsoft/Phi-4-multimodal-inst

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

/tmp/ipykernel_55/2158367193.py:125: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



VALIDATION metrics:
  accuracy: 0.993640
  f1_binary: 0.993921
  f1_macro: 0.993626
  precision: 0.993845
  recall: 0.993997


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



TEST_SAMPLE (labeled subset) metrics:
  accuracy: 0.270000
  f1_binary: 0.343525
  f1_macro: 0.260727
  precision: 0.214848
  recall: 0.856502

Best threshold on TEST_SAMPLE for F1(binary): t=0.10, F1=0.345568



Saved: /kaggle/working/submission_argmax.csv
   ID  label
0   0      0
1   2      0
2   5      1
3   6      0
4   7      0

Saved: /kaggle/working/submission_thresholded.csv
   ID  label
0   0      0
1   2      0
2   5      1
3   6      0
4   7      0

Done. Submit one (or both) of these:
 - /kaggle/working/submission_argmax.csv
 - /kaggle/working/submission_thresholded.csv
